In [ ]:
import json
import os
import random

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely.geometry as sg

from xyzservices import TileProvider

RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

def distance_and_duration(gdf: gpd.GeoDataFrame) -> pd.Series:
    distance = sg.LineString(gdf.geometry).length
    duration = gdf["timestamp_os"].max() - gdf["timestamp_os"].min()
    speed = fps = mpf = None
    if duration.seconds > 0:
        speed = distance / duration.seconds
        fps = len(gdf) / duration.seconds
        mpf = distance / len(gdf)
    result = {
        "distance (km)": distance / 1000,
        "duration": pd.Timedelta(seconds=duration.seconds),
        "speed (m/s)": speed,
        "FPS": fps,
        "MPF": mpf,
    }
    return pd.Series(result)

In [ ]:
data_folder = "../datasets/experiments/zwerfafval"

metadata_file = "EDW_training_beeldinwinning_21_04_2026.json"
imu_file = "inwinning.npz"
frames_folder = "frames"

In [ ]:
with open(os.path.join(data_folder, metadata_file), 'r') as f:
    json_data = json.load(f)

metadata_df = pd.DataFrame(data=json_data['gps_log']['data'])
metadata_df["timestamp"] = pd.to_datetime(metadata_df["timestamp"], unit="s")
metadata_df["timestamp_utc"] = pd.to_datetime(metadata_df["timestamp_utc"], unit="s", utc=True)
metadata_df["timestamp_os"] = pd.to_datetime(metadata_df["timestamp_os"], unit="ns")

metadata_gdf = gpd.GeoDataFrame(
    metadata_df, 
    geometry=[sg.Point(lon, lat) for lat, lon in zip(metadata_df["latitude"], metadata_df["longitude"])],
    crs=LAT_LON_CRS
)

metadata_gdf = metadata_gdf[['timestamp_os', 'speed_kmh', 'accuracy_speed', 'geometry']]

del json_data, metadata_df

In [ ]:
distance_and_duration(metadata_gdf.to_crs(RD_CRS))

In [ ]:
map = metadata_gdf.to_crs(RD_CRS).explore(tiles=ams_tile_provider)
map.save(os.path.join(data_folder, "inwinning.html"))

In [ ]:
imu = np.load(os.path.join(data_folder, imu_file))

recording_start = pd.to_datetime(imu["timestamps"][0], unit="s")
recording_end = pd.to_datetime(imu["timestamps"][-1], unit="s")

del imu

In [ ]:
fps = 1

frames = sorted([
    file for file in os.listdir(os.path.join(data_folder, frames_folder))
    if file.endswith(".jpg")
])

first_frame = recording_start + pd.Timedelta(seconds=(1/fps/2))  # FFMPEG extracts frames mid-interval, so start at 0.25s for 2 FPS
frame_timestamps = pd.date_range(first_frame, recording_end, freq=pd.Timedelta(seconds=1/fps))

frames_df = pd.DataFrame(
    data = {
        "file_name": frames,
        "frame_timestamp": pd.to_datetime(frame_timestamps)
    }
)

del frames, frame_timestamps

In [ ]:
merged_df = pd.merge_asof(frames_df, metadata_gdf, left_on="frame_timestamp", right_on="timestamp_os", direction='nearest')

merged_df.insert(
    loc=3, 
    column="time_diff", 
    value = abs(merged_df["frame_timestamp"] - merged_df["timestamp_os"]).dt.total_seconds()
)

merged_gdf = gpd.GeoDataFrame(
    merged_df, 
    crs=LAT_LON_CRS
)

del merged_df

In [ ]:
merged_gdf.to_file(os.path.join(data_folder, "frames_gdf.gpkg"), driver="GPKG")

In [ ]:
# Accept frame when speed is larger than threshold

min_speed = 3.0

merged_gdf["accept"] = merged_gdf["speed_kmh"] >= min_speed

In [ ]:
# Select frame when distance to previous selected is larger than threshold

min_distance = 5.0

points = merged_gdf[merged_gdf["accept"]]["geometry"].to_crs(RD_CRS)

previous = points.iloc[0]

selection = [False]*len(points)
selection[0] = True

for i, point in enumerate(points.iloc[1:]):
    if previous.distance(point) >= min_distance:
        previous = point
        selection[i+1] = True

merged_gdf["select_5m"] = False
merged_gdf.loc[merged_gdf["accept"], "select_5m"] = selection

In [ ]:
# Split data in groups of ``group_size`` consecutive frames, and select `n_groups_selected` groups

group_size = 20
n_groups_selected = 50

selected = merged_gdf[merged_gdf["select_5m"]].copy()


n_groups = int(np.ceil(len(selected) / group_size))
groups = [i for i in range(n_groups) for _ in range(group_size)]
selected.loc[:, "group"] = groups[0:len(selected)]

group_sample = random.sample(range(n_groups), n_groups_selected)

dataselectie_gdf = selected[selected["group"].isin(set(group_sample))]

In [ ]:
len(dataselectie_gdf)

In [ ]:
dataselectie_gdf.plot()

In [ ]:
map = dataselectie_gdf.to_crs(RD_CRS).explore(tiles=ams_tile_provider)
map.save(os.path.join(data_folder, "data_selectie.html"))

In [ ]:
import shutil

source_folder = os.path.join(data_folder, "frames")
dest_folder = os.path.join(data_folder, "frames_selected_1000")


for file in dataselectie_gdf["file_name"]:
    shutil.copy(os.path.join(source_folder, file), dest_folder)

In [ ]:
dataselectie_gdf.to_file(os.path.join(data_folder, "dataselectie_1000.gpkg"), driver="GPKG")